# AI-data-map: System-1 tiny model training (Colab, fallback)

Fallback for the Kaggle primary notebook. Same pipeline: validate shards, train, export to ONNX, run the eval gates, push weights to the Hugging Face model repo.

Setup: Runtime > Change runtime type > GPU. Add an `HF_TOKEN` secret (Colab Secrets panel, notebook access on) with write access to the model repo.

In [ ]:
!pip install --quiet torch onnx onnxruntime huggingface_hub datasets

In [ ]:
import torch
print("cuda available:", torch.cuda.is_available())
assert torch.cuda.is_available(), "enable a GPU runtime first (Runtime > Change runtime type)"

In [ ]:
import os, glob
from huggingface_hub import snapshot_download
from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

DATASET_REPO = "movahedi-ca/ai-data-map-teacher"  # Phase 4 publishes here
MODEL_REPO = "movahedi-ca/s1-workflow-tiny"
data_dir = snapshot_download(repo_id=DATASET_REPO, repo_type="dataset")
shards = sorted(glob.glob(os.path.join(data_dir, "shards", "*.jsonl")))
print(f"{len(shards)} shards from {data_dir}")
assert shards, "no shards found; check the dataset repo"
with open("/content/shards.txt", "w") as fh:
    fh.write(" ".join(shards))

In [ ]:
!git clone --quiet https://github.com/movahedi-ca/AI-data-map.git /content/AI-data-map
%cd /content/AI-data-map/training

In [ ]:
!python -c "
from data import load_shards
paths = open('/content/shards.txt').read().split()
steps = load_shards(paths)
from collections import Counter
print(len(steps), 'steps;', Counter(s['domain'] for s in steps))
"

In [ ]:
!python train.py --shards $(cat /content/shards.txt) --out /content/out --epochs 12 --batch 256 --lr 3e-4

In [ ]:
!python export_onnx.py --checkpoint /content/out/checkpoint.pt --out /content/out/model.onnx

In [ ]:
!python eval.py --checkpoint /content/out/checkpoint.pt --shards $(cat /content/shards.txt) \
  --report /content/out/eval-report.json --train-domains data-mapping \
  --indomain-gate 0.85 --ood-gate 0.60 --perm-tolerance 0.10

In [ ]:
from huggingface_hub import HfApi
api = HfApi()
api.create_repo(repo_id=MODEL_REPO, exist_ok=True)
for f in ["model.onnx", "checkpoint.pt", "config.json", "eval-report.json"]:
    api.upload_file(path_or_fileobj=f"/content/out/{f}", path_in_repo=f, repo_id=MODEL_REPO)
api.upload_file(path_or_fileobj="MODEL-CARD.md", path_in_repo="README.md", repo_id=MODEL_REPO)
print("pushed to", MODEL_REPO)